In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Controls — immediately after Drive mount; safe for Runtime → Run all
DRIVE_ROOT='/content/drive/MyDrive/OpenPlaque'
OUTPUT_ROOT=DRIVE_ROOT + '/Longitudinal_Plaque_PCAT_Fusion_v1'


# OpenPlaque — Longitudinal plaque-confidence + PCAT fusion

Research use only. This workflow uses only the validated longitudinal component of the canonical curved-series → source-CCTA registration. Plaque quantities remain native curved-series support counts; they are not source-space mm³, validated TPV, or a 3-D plaque mask. RCA plaque support is aligned with the locked OpenPlaque PCAT attenuation profile over 10–50 mm. LAD plaque support is mapped to canonical source arc only; no LAD PCAT fusion is attempted.

The native plaque-confidence profile is regenerated directly from each saved 0–5 vote volume using the vessel-specific validated registration axis. This avoids the earlier 512×512 tie-break ambiguity in the confidence-atlas longitudinal-axis heuristic.


In [ ]:
!pip -q install pandas numpy matplotlib pytest SimpleITK
import shutil, sys
from pathlib import Path
repo=Path('/content/OpenPlaque')
if repo.exists(): shutil.rmtree(repo)
!git clone -q --depth 1 --branch longitudinal-plaque-pcat-fusion-from-main https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
sys.path.insert(0,'/content/OpenPlaque/src')


In [ ]:
# Unit checks before reading Drive data
!cd /content/OpenPlaque && PYTHONPATH=/content/OpenPlaque/src pytest -q tests/test_longitudinal_plaque_pcat_fusion.py tests/test_longitudinal_plaque_pcat_fusion_corrected.py


In [ ]:
from openplaque.longitudinal_plaque_pcat_fusion_corrected import run
result = run(drive_root=DRIVE_ROOT, output_root=OUTPUT_ROOT)
print('STATUS:', result['status'])
print('SCIENTIFIC STATUS:', result['scientific_status'])
for vessel, v in result['summary']['registration'].items():
    print(vessel, 'longitudinal score=', round(v['longitudinal_score'],3), 'gradient=', round(v['longitudinal_gradient_corr'],3), 'axis=', v['long_axis'], 'mapped rows=', v['mapped_native_rows'])
print('RCA overlap:', result['summary']['rca_overlap_summary'])
print('REPORT:', result['report'])
print('ZIP:', result['zip'])
print('Drive search: https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_LONGITUDINAL_PLAQUE_PCAT_FUSION_REPORT_BACK.zip')
